# 100-neuron excitatory/inhibitory network source-to-field/readout workflow

This tutorial scales the two-neuron E/I workflow to a population-level 100-neuron network.

**What you will learn:**
- Configure a 100-neuron balanced E/I network (75 excitatory, 25 inhibitory)
- Simulate population dynamics with JAX-native vectorization
- Apply the eight standard proxy readout operators to population activity
- Analyze population metrics and field projections
- Understand the computational scope and readout framework for larger networks

**Prerequisites:** Familiarity with [Tutorial 1: Single-neuron multimodal](01_single_neuron_multimodal.ipynb) and [Tutorial 2: Two-neuron E/I](02_two_neuron_ei_multimodal.ipynb)

**Estimated time:** 10–15 minutes

In [ ]:
!pip install jaxfne

In [ ]:
import jaxfne
print(f"jaxfne version: {jaxfne.__version__}")

## Imports

Import JAX and jaxfne components for population-level simulation.

In [ ]:
import jax
import jax.numpy as jnp
from jaxfne.fields import (
    spk_probe,
    vm_probe,
    source_probe,
    lfp_proxy_probe,
    csd_proxy_probe,
    eeg_proxy_probe,
    meg_proxy_probe,
    emm_proxy_probe,
)

# Verify JAX is available
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## Configuration

Set up a 100-neuron balanced E/I network with 75 excitatory and 25 inhibitory Izhikevich neurons.

In [ ]:
# Create 100-neuron E/I configuration
# 75 excitatory, 25 inhibitory, 100ms simulation, 0.1ms dt
cfg = (
    jaxfne.configuration()
    .network(
        name="network_100_ei",
        kind="balanced_ei_population",
        n=100,
        cell_types={"E": 0.75, "I": 0.25},
    )
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(
        domain="laminar_column",
        conductivity="proxy",
        boundary="declared_proxy",
        gauge="mean_zero",
    )
    .probe(
        name="multimodal_100_ei",
        modes=[
            "spikes",
            "V_m",
            "source",
            "phi_e",
            "J_e",
            "CSD",
            "LFP",
        ],
    )
)

# Construct the JAX-native model
model = jaxfne.construct(cfg)
print("100-neuron E/I model constructed successfully.")
print(f"Network: 100 neurons (75 excitatory, 25 inhibitory)")

## Run Simulation

Simulate 100 milliseconds of population E/I activity.
JAX vectorization ensures efficient CPU computation across all 100 neurons.

In [ ]:
# Define simulation parameters with deterministic seed
sim = jaxfne.simulation(duration_ms=100.0, dt_ms=0.1, seed=42)

# Run the simulation
signals = model.simulate(sim)

print(f"Simulation complete.")
print(f"Spike matrix shape: {signals.spikes.shape}  [time, neurons]")
print(f"Voltage shape: {signals.V_m.shape}         [time, neurons]")
if hasattr(signals, 'field') and signals.field is not None:
    print(f"LFP proxy shape: {signals.field.lfp_proxy.shape}    [time, contacts]")

## Apply Readout Operators

Apply all eight standard proxy readout operators to the 100-neuron network signals.
These operators extract different modalities from the population activity.

In [ ]:
# Apply all eight probe operators
spk_readout = spk_probe(signals.spikes)
vm_readout = vm_probe(signals.V_m)

# Source and field-derived operators
source_readout = (
    source_probe(signals.sources[0]) if signals.sources is not None else None
)
lfp_readout = (
    lfp_proxy_probe(signals.field.lfp_proxy)
    if signals.field is not None
    else None
)
csd_readout = (
    csd_proxy_probe(signals.field.csd_proxy)
    if signals.field is not None
    else None
)

# EEG, MEG, EMM operators use field signal
field_signal = (
    signals.field.lfp_proxy if signals.field is not None else signals.V_m
)
eeg_readout = eeg_proxy_probe(field_signal)
meg_readout = meg_proxy_probe(field_signal)
emm_readout = emm_proxy_probe(field_signal)

# Construct probe_report from all operators
probe_report = {
    "spk": spk_readout.report,
    "vm": vm_readout.report,
}
if source_readout is not None:
    probe_report["source"] = source_readout.report
if lfp_readout is not None:
    probe_report["lfp_proxy"] = lfp_readout.report
if csd_readout is not None:
    probe_report["csd_proxy"] = csd_readout.report

probe_report.update({
    "eeg_proxy": eeg_readout.report,
    "meg_proxy": meg_readout.report,
    "emm_proxy": emm_readout.report,
})

print(f"All eight probe operators applied to 100-neuron network:")
for op_name in probe_report.keys():
    print(f"  ✓ {op_name}")

## Analyze Population Dynamics

Examine population-level metrics: spike counts, firing rates, and voltage statistics.

In [ ]:
# Analyze population activity
total_spikes = jnp.sum(signals.spikes)
excitatory_spikes = jnp.sum(signals.spikes[:, :75])  # First 75 are excitatory
inhibitory_spikes = jnp.sum(signals.spikes[:, 75:])  # Last 25 are inhibitory

# Population spike rate (spikes per timestep, all neurons)
pop_spike_rate_per_ts = jnp.mean(jnp.sum(signals.spikes, axis=1))

# Voltage statistics
voltage_mean = jnp.mean(signals.V_m)
voltage_std = jnp.std(signals.V_m)
voltage_min = jnp.min(signals.V_m)
voltage_max = jnp.max(signals.V_m)

print("Population activity (100 neurons):")
print(f"  Total spikes: {int(total_spikes)}")
print(f"  Excitatory (75 neurons): {int(excitatory_spikes)} spikes")
print(f"  Inhibitory (25 neurons): {int(inhibitory_spikes)} spikes")
print(f"  Population spike rate: {float(pop_spike_rate_per_ts):.6f} spikes/timestep")
print(f"\nVoltage statistics (all neurons):")
print(f"  Mean: {float(voltage_mean):.2f} mV")
print(f"  Std: {float(voltage_std):.2f} mV")
print(f"  Min: {float(voltage_min):.2f} mV")
print(f"  Max: {float(voltage_max):.2f} mV")

## Output Bundle Inspection

Examine the reproducible output bundle: manifest (metadata), probe reports, and metrics.

In [ ]:
# Get the manifest (full pipeline metadata)
manifest = model.manifest(signals=signals)

# Compute population metrics
metrics = {
    "n_neurons": 100,
    "n_excitatory": 75,
    "n_inhibitory": 25,
    "spike_count_total": int(total_spikes),
    "spike_count_excitatory": int(excitatory_spikes),
    "spike_count_inhibitory": int(inhibitory_spikes),
    "spike_rate_hz": float(pop_spike_rate_per_ts * 10000.0),  # Convert to Hz (0.1ms dt)
    "Vm_mean_mV": float(voltage_mean),
    "Vm_std_mV": float(voltage_std),
}

# Display manifest keys
print("Manifest keys (pipeline metadata):")
manifest_keys = list(manifest.keys())[:5]
for key in manifest_keys:
    print(f"  - {key}")
if len(manifest.keys()) > 5:
    print(f"  ... and {len(manifest.keys()) - 5} more")

# Display population metrics
print(f"\nPopulation metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

## Readout Summary

Verify the scope metadata from the manifest.
These fields indicate the computational framework and any validation present.
All readouts remain computational proxies: no biological mechanism claims, no empirical validation.

In [ ]:
# Extract and display scope metadata
scope_summary = {
    "claim_level": manifest.get("claim_level"),
    "field_claim_level": manifest.get("field_claim_level"),
    "field_solver_status": manifest.get("field_solver_status"),
    "source_calibration_status": manifest.get("source_calibration_status"),
    "physical_amplitude_claim_allowed": manifest.get("physical_amplitude_claim_allowed"),
}

print(f"Readout summary (immutable, computational framework):")
for key, value in scope_summary.items():
    print(f"  {key}: {value}")

print(f"\n✓ Physical amplitude claims allowed: {scope_summary['physical_amplitude_claim_allowed']}")
print(f"✓ All eight operators executed successfully on 100-neuron population.")
print(f"✓ Output bundle is JSON-serializable and reproducible.")
print(f"\nScope: Computational scaffold for population-level E/I learning, not empirically validated.")

## Summary

You have completed the 100-neuron E/I multimodal source-to-field/readout workflow:

1. **Configuration:** Defined a 100-neuron balanced E/I network (75E, 25I) with Izhikevich emitters and laminar field
2. **Simulation:** Ran 100ms of JAX-native population dynamics (1000 timesteps, 100 neurons)
3. **Readouts:** Applied eight standard proxy operators to population signals
4. **Analysis:** Examined population metrics, field projections, and scope metadata

### Scaling observations

- JAX vmap handles 100 neurons efficiently on CPU
- Population spike rates and field projections scale smoothly
- All eight operators work identically at population scale
- Output bundles remain JSON-serializable and reproducible

### What's next

- **[Guides: Probe operators](../guides/probe_operators.md)** — Details on each readout operator
- **[API Reference](../api/index.md)** — Full function and class documentation
- **[Tensor-field workflows](../guides/tensor_field_workflows.md)** — Deep dive on field mathematics
- **[Calibration](../guides/calibration.md)** — Preparing workflows for future empirical validation

### Questions?

File an issue or discuss on GitHub: https://github.com/HNXJ/jaxfne